# TMDB 영화 데이터셋 전처리 실습 — 강사용

> 🔴 **강사 전용 노트북입니다. 학생에게 배포하지 마세요.**

## 강의 운영 가이드

### 전체 흐름 (권장 시간: 120분)
| 단계 | 내용 | 시간 | 강의 방식 |
|---|---|---|---|
| Step 1-2 | 라이브러리 & 데이터 로드 | 10분 | 시연 후 학생 따라하기 |
| Step 3 | 보조 함수 3개 정의 | 25분 | TODO 2~4, 핵심 개념 설명 |
| Step 4 | 파생 변수 생성 | 20분 | TODO 5, lambda 패턴 설명 |
| Step 5 | 결측값 처리 | 10분 | TODO 6, 데이터 품질 토론 |
| Step 6 | 감독·배우 인지도 | 25분 | TODO 7~8, groupby·explode 설명 |
| Step 7 | 마케팅 점수 | 15분 | TODO 9, Min-Max 수식 판서 |
| Step 8-9 | 저장 & EDA | 15분 | TODO 10~11, 탐색 자유 시간 |

### 강조할 핵심 개념
- `literal_eval` vs `json.loads` 차이 (Step 3)
- `explode()` 를 이용한 1:N 데이터 처리 패턴 (Step 6 TODO 8)
- Min-Max Scaling의 의미와 수식 (Step 7)
- `encoding='utf-8-sig'` 이유 — 엑셀 한글 깨짐 방지 (Step 8)


## Step 1: 라이브러리 임포트 & 파일 경로 설정

**🔴 강사 노트**: `re` 모듈은 이번 실습에서 직접 사용하지 않지만,
이후 텍스트 전처리 확장 시 필요하므로 미리 임포트해 둡니다.
학생들이 '왜 임포트하냐'고 물으면 이 점을 설명하세요.


In [ ]:
# ✅ [제공 코드] 필요한 라이브러리 임포트
import pandas as pd
import numpy as np
import re                       # 정규표현식 (텍스트 패턴 찾기)
from ast import literal_eval    # 문자열 → 파이썬 리스트/딕셔너리 안전 변환

# 파일 경로 정의
movies_path  = 'tmdb_5000_movies.csv'
credits_path = 'tmdb_5000_credits.csv'
output_path  = 'movie_box_office_prediction_tmdb_like.csv'

print('✅ 라이브러리 로드 완료')


## Step 2: 원본 데이터 로드 & 병합

**🔴 강사 노트**: merge 방식 설명 포인트
- `how='left'`: movies 기준으로 credits를 붙임 → credits에 없는 영화도 유지
- `on='title'`: 영화 제목이 완전히 일치하는 경우만 병합. 오타가 있으면 NaN 발생
- 병합 후 shape 변화를 학생들에게 직접 확인시키세요.

**예상 질문**: "inner join이랑 left join 차이가 뭔가요?"
→ inner: 양쪽 모두 있는 것만 / left: 왼쪽(movies) 전체 + 오른쪽(credits) 매칭되는 것


In [ ]:
# [TODO 1 정답] 두 CSV를 로드하고 병합
movies  = pd.read_csv(movies_path)
credits = pd.read_csv(credits_path)

print(f'movies  shape: {movies.shape}')
print(f'credits shape: {credits.shape}')

df = movies.merge(credits, on='title', how='left')

print(f'\n병합 후 shape: {df.shape}')
df.head(3)


## Step 3: 전처리 보조 함수 정의

**🔴 강사 노트 — literal_eval 개념 설명 (5분 판서)**

```python
# CSV에 저장된 실제 값 (문자열 타입)
raw = '[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}]'
type(raw)  # → str

# literal_eval 적용 후 (리스트 타입)
from ast import literal_eval
parsed = literal_eval(raw)
type(parsed)  # → list
parsed[0]     # → {'id': 28, 'name': 'Action'}
```

**json.loads와 차이**: `literal_eval`은 파이썬 리터럴(True/False/None 등)도 처리 가능.
보안 측면에서도 eval() 보다 안전합니다.


In [ ]:
# [TODO 2 정답] safe_literal_eval — 문자열 → 파이썬 객체 변환
def safe_literal_eval(x):
    """
    CSV 안의 텍스트가 '[{"id": 1, "name": "Action"}]' 같은 문자열일 때,
    이를 실제 파이썬 리스트 객체로 변환해주는 안전한 함수
    """
    if pd.isna(x):
        return []
    try:
        return literal_eval(x)
    except:
        return []

# 5개 컬럼에 일괄 적용
for col in ['genres', 'keywords', 'production_companies', 'cast', 'crew']:
    if col in df.columns:
        df[col] = df[col].apply(safe_literal_eval)

# 🔴 강사 확인 포인트: 변환 전후 타입 비교를 학생에게 보여주세요
print('genres 컬럼 첫 번째 값 타입:', type(df['genres'].iloc[0]))
print('genres 첫 번째 값:', df['genres'].iloc[0][:2], '...')


In [ ]:
# [TODO 3 정답] extract_names — 딕셔너리 리스트에서 'name' 값만 추출
def extract_names(items, top_n=None):
    """
    리스트 안에 딕셔너리 구조(예: [{'name': '액션'}, {'name': '스릴러'}])에서
    'name'에 해당하는 글자들만 뽑아내는 함수
    top_n을 지정하면 상위 N개만 추출
    """
    if not isinstance(items, list):
        return []
    names = [
        item.get('name', '')
        for item in items
        if isinstance(item, dict) and item.get('name')
    ]
    return names[:top_n] if top_n else names

# 테스트
sample = [{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}]
print('전체 추출:', extract_names(sample))
print('상위 2개:', extract_names(sample, top_n=2))


In [ ]:
# [TODO 4 정답] extract_director — 제작진 리스트에서 감독 이름 추출
def extract_director(crew_list):
    """
    제작진(crew) 리스트를 돌면서 job이 'Director'인 사람의 이름만 반환
    """
    if not isinstance(crew_list, list):
        return np.nan
    for person in crew_list:
        if isinstance(person, dict) and person.get('job') == 'Director':
            return person.get('name')
    return np.nan

# 테스트
sample_crew = [
    {'job': 'Producer',  'name': 'John Smith'},
    {'job': 'Director',  'name': 'James Cameron'},
    {'job': 'Screenplay','name': 'Jane Doe'}
]
print('감독 추출 결과:', extract_director(sample_crew))

# 🔴 강사 심화 질문: "감독이 두 명인 영화가 있을 수 있을까요? 그럼 어떻게 처리해야 할까요?"


## Step 4: 파생 변수 생성

**🔴 강사 노트 — lambda 패턴 설명**

학생들이 자주 헷갈리는 부분:
```python
# 이 두 코드는 완전히 동일합니다
df['genre'] = df['genres'].apply(lambda x: extract_names(x, 1)[0] if len(extract_names(x, 1)) > 0 else 'Unknown')

# 함수로 풀어쓰면
def get_first_genre(x):
    names = extract_names(x, 1)
    return names[0] if len(names) > 0 else 'Unknown'
df['genre'] = df['genres'].apply(get_first_genre)
```

**`pd.to_numeric(..., errors='coerce')` 설명**:
- `errors='coerce'`: 변환 불가능한 값(문자열 등) → NaN으로 처리
- `errors='raise'`: 변환 실패 시 오류 발생 (기본값)
- `errors='ignore'`: 변환 실패 시 원본 값 유지


In [ ]:
# [TODO 5 정답] 파생 변수 6개 생성

# (1) 장르: 첫 번째 장르 1개
df['genre'] = df['genres'].apply(
    lambda x: extract_names(x, 1)[0] if len(extract_names(x, 1)) > 0 else 'Unknown'
)

# (2) 감독 이름
df['director_name'] = df['crew'].apply(extract_director)

# (3) 주연 배우 3명 리스트
df['cast_names'] = df['cast'].apply(lambda x: extract_names(x, 3))

# (4) 수치형 컬럼 강제 변환
for col in ['runtime', 'budget', 'revenue', 'vote_average', 'vote_count', 'popularity']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# (5) 개봉일 → 월/연도 분리
df['release_date']  = pd.to_datetime(df['release_date'], errors='coerce')
df['release_month'] = df['release_date'].dt.month
df['release_year']  = df['release_date'].dt.year

# (6) 참여 제작사 수
df['production_company_size'] = df['production_companies'].apply(
    lambda x: len(extract_names(x))
)

print('파생 변수 생성 완료')
print(df[['title','genre','director_name','cast_names','release_year','production_company_size']].head(3))


## Step 5: 결측값 & 이상값 제거

**🔴 강사 노트 — 데이터 품질 토론 포인트 (5분)**

학생들에게 먼저 물어보세요:
> *"budget=0인 영화는 왜 제거해야 할까요? 정말 제작비 0원인 영화일 수도 있지 않나요?"*

- TMDB 데이터에서 0은 대부분 **'데이터 미입력'** 을 의미합니다.
- 실제 독립 저예산 영화는 0이 아닌 소액으로 등록되어 있습니다.
- **도메인 지식**이 전처리 기준을 결정한다는 점을 강조하세요.

**`.fillna(0)` 사용 이유**: 비교 연산(`> 0`) 전에 NaN을 0으로 채워야
NaN 비교(`NaN > 0 → False`)로 인한 의도치 않은 행 제거를 막을 수 있습니다.


In [ ]:
# [TODO 6 정답] 비정상 데이터(budget/revenue/runtime = 0 or NaN) 제거
before = len(df)

df = df[
    (df['budget'].fillna(0)  > 0) &
    (df['revenue'].fillna(0) > 0) &
    (df['runtime'].fillna(0) > 0)
].copy()

after = len(df)
print(f'제거 전: {before}행  →  제거 후: {after}행  (제거: {before - after}행)')
print(f'\n결측 요약:\n{df[["budget","revenue","runtime","director_name"]].isnull().sum()}')


## Step 6: 감독 & 배우 인지도(Popularity) 점수 계산

**🔴 강사 노트 — groupby → merge 패턴 설명**

```
[df 원본]
title        director_name    popularity
Avatar       James Cameron    150.4
Titanic      James Cameron    200.1
Inception    Christopher N.   80.2

[groupby 결과: director_pop]
director_name         director_popularity_raw
James Cameron         175.25  ← (150.4 + 200.1) / 2
Christopher Nolan     80.2

[merge 후: df]
title        director_name    director_popularity_raw
Avatar       James Cameron    175.25
Titanic      James Cameron    175.25  ← 동일 감독 → 동일 점수
Inception    Christopher N.   80.2
```


In [ ]:
# [TODO 7 정답] 감독 인지도 점수 계산 & 병합
director_pop = (
    df.groupby('director_name')['popularity']
    .mean()
    .reset_index()
    .rename(columns={'popularity': 'director_popularity_raw'})
)

df = df.merge(director_pop, on='director_name', how='left')

print('감독 인지도 Top 5:')
print(director_pop.sort_values('director_popularity_raw', ascending=False).head(5))


**🔴 강사 노트 — explode() 개념 설명 (5분 판서)**

```
[explode 전]
title     cast_names
Avatar    ['Sam W.', 'Zoe S.', 'Sigourney W.']
Titanic   ['Leo D.', 'Kate W.']

[explode('cast_names') 후]
title     cast_name
Avatar    Sam W.
Avatar    Zoe S.
Avatar    Sigourney W.
Titanic   Leo D.
Titanic   Kate W.
```

1:N 관계 데이터를 처리하는 핵심 패턴입니다.
SQL의 UNNEST / LATERAL JOIN과 동일한 개념임을 설명하세요.


In [ ]:
# [TODO 8 정답] 배우 인지도 점수 계산 & 병합

# (1) cast_names 리스트를 행으로 펼치기
cast_exploded = df[['title', 'popularity', 'cast_names']].explode('cast_names')
cast_exploded = cast_exploded.rename(columns={'cast_names': 'cast_name'})
cast_exploded = cast_exploded.dropna(subset=['cast_name'])

# (2) 배우별 평균 인지도 계산
actor_pop = (
    cast_exploded.groupby('cast_name')['popularity']
    .mean()
    .reset_index()
    .rename(columns={'popularity': 'actor_popularity_raw'})
)
cast_exploded = cast_exploded.merge(actor_pop, on='cast_name', how='left')

# (3) 영화별 평균 배우 인지도
movie_cast_pop = (
    cast_exploded.groupby('title')['actor_popularity_raw']
    .mean()
    .reset_index()
    .rename(columns={'actor_popularity_raw': 'cast_popularity_raw'})
)

df = df.merge(movie_cast_pop, on='title', how='left')

print('배우 인지도 Top 5:')
print(actor_pop.sort_values('actor_popularity_raw', ascending=False).head(5))


## Step 7: 마케팅 점수(marketing_score) 생성

**🔴 강사 노트 — Min-Max Scaling 판서 (3분)**

```
score = (x - x_min) / (x_max - x_min) × 100

예시: popularity = [10, 50, 100]
  x_min = 10, x_max = 100
  10  → (10-10)/(100-10) × 100 = 0점
  50  → (50-10)/(100-10) × 100 = 44.4점
  100 → (100-10)/(100-10) × 100 = 100점
```

**왜 log1p를 쓰나요?** (vote_count, budget에 적용)
- 값의 범위가 너무 클 때 (예: budget 1000 ~ 300,000,000) 로그로 스케일을 줄여
  소수의 블록버스터가 점수를 독식하는 것을 방지합니다.
- `log1p(x) = log(1+x)` → x=0일 때 log(0) 오류를 방지합니다.


In [ ]:
# [TODO 9 정답] Min-Max Scaling & marketing_score 계산
def minmax_series(s):
    """시리즈를 0~100 범위로 Min-Max 정규화"""
    s     = s.fillna(s.median())
    min_v = s.min()
    max_v = s.max()
    if max_v == min_v:
        return pd.Series([50.0] * len(s), index=s.index)
    return (s - min_v) / (max_v - min_v) * 100

df['popularity_scaled']  = minmax_series(df['popularity'])
df['vote_count_scaled']  = minmax_series(np.log1p(df['vote_count'].fillna(0)))
df['budget_scaled']      = minmax_series(np.log1p(df['budget'].fillna(0)))

df['marketing_score'] = (
    df['popularity_scaled']  * 0.5 +
    df['vote_count_scaled']  * 0.3 +
    df['budget_scaled']      * 0.2
)

print('marketing_score 통계:')
print(df['marketing_score'].describe().round(2))

# 🔴 강사 심화 질문: "가중치(0.5, 0.3, 0.2)를 바꾸면 어떤 영화가 올라오고 내려갈까요?"


## Step 8: 최종 컬럼 선택 & CSV 저장

**🔴 강사 노트**: `encoding='utf-8-sig'` 필수 이유
- `utf-8`: 기본 UTF-8. 엑셀에서 열면 한글이 깨지는 경우 있음
- `utf-8-sig`: BOM(Byte Order Mark) 포함 UTF-8. 엑셀이 자동으로 UTF-8로 인식
- 실무에서 데이터를 비개발자와 공유할 때 반드시 확인해야 할 포인트입니다.


In [ ]:
# ✅ [제공 코드] 최종 컬럼 정의
final_cols = [
    'title', 'genre', 'director_name', 'cast_names',
    'budget', 'revenue', 'runtime',
    'vote_average', 'vote_count', 'popularity',
    'release_month', 'release_year', 'production_company_size',
    'director_popularity_raw', 'cast_popularity_raw',
    'marketing_score'
]
final_cols_exist = [c for c in final_cols if c in df.columns]
df_final = df[final_cols_exist].copy()
print(f'최종 데이터셋: {df_final.shape[0]}행 × {df_final.shape[1]}열')
df_final.head()


In [ ]:
# [TODO 10 정답] CSV 저장
df_save = df_final.copy()
df_save['cast_names'] = df_save['cast_names'].apply(
    lambda x: ', '.join(x) if isinstance(x, list) else x
)

df_save.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'✅ 저장 완료: {output_path}')
print(f'   최종 shape: {df_save.shape}')
print(f'\n컬럼 목록:')
for c in df_save.columns:
    print(f'  - {c}: {df_save[c].dtype}  (결측: {df_save[c].isnull().sum()}개)')


## Step 9: 데이터 탐색 (EDA)

**🔴 강사 노트**: 학생들이 TODO 11을 완성하면,
아래 심화 질문으로 토론을 유도하세요.

1. *"가장 흥행한 장르는 무엇인가요? 그 이유는?"*
2. *"marketing_score가 높은데 revenue가 낮은 영화가 있나요? 왜 그럴까요?"*
3. *"director_popularity_raw와 revenue 사이에 상관관계가 있을까요?"*


In [ ]:
# [TODO 11 정답] EDA — 기본 통계 확인
df_check = pd.read_csv(output_path)

# Q1. 장르별 영화 수 Top 10
print('=== 장르별 영화 수 Top 10 ===')
print(df_check['genre'].value_counts().head(10))

# Q2. 수익 Top 5 영화
print('\n=== 수익 Top 5 영화 ===')
print(df_check.nlargest(5, 'revenue')[['title', 'revenue', 'budget', 'genre']])

# Q3. marketing_score Top 5 영화
print('\n=== 마케팅 점수 Top 5 영화 ===')
print(df_check.nlargest(5, 'marketing_score')[['title', 'marketing_score', 'director_name']])


## 💡 심화 도전 (학생용 과제)
1. `revenue / budget`으로 **ROI(투자수익률)** 컬럼을 추가하고 ROI 상위 10개 영화를 확인하세요.
2. `release_month`별 평균 `revenue`를 막대그래프로 시각화하세요.
3. `director_popularity_raw`가 높은 감독과 낮은 감독의 평균 `revenue` 차이를 비교하세요.
4. `vote_average` 기준 8.0 이상 영화만 추출하여 장르 분포를 파이차트로 그려보세요.


In [ ]:
# ✅ [강사용 심화 정답 예시]

# 심화 1: ROI 계산
df_check['ROI'] = df_check['revenue'] / df_check['budget']
print('=== ROI Top 10 ===')
print(df_check.nlargest(10, 'ROI')[['title', 'ROI', 'budget', 'revenue']].round(2))

# 심화 2: 월별 평균 수익
monthly = df_check.groupby('release_month')['revenue'].mean().reset_index()
print('\n=== 월별 평균 수익 (단위: 억) ===')
monthly['revenue_億'] = (monthly['revenue'] / 1e8).round(1)
print(monthly.to_string(index=False))

# 심화 3: 감독 인지도 상위/하위 집단 수익 비교
median_dir_pop = df_check['director_popularity_raw'].median()
high_dir = df_check[df_check['director_popularity_raw'] >= median_dir_pop]['revenue'].mean()
low_dir  = df_check[df_check['director_popularity_raw'] <  median_dir_pop]['revenue'].mean()
print(f'\n=== 감독 인지도 상위 평균 수익: {high_dir/1e8:.1f}억 vs 하위: {low_dir/1e8:.1f}억 ===')


## 📋 수업 마무리 체크리스트

- [ ] `literal_eval` vs `eval` 차이 설명 완료
- [ ] `merge(how='left')` 의미 설명 완료
- [ ] `explode()` 패턴 판서/시연 완료
- [ ] Min-Max Scaling 수식 판서 완료
- [ ] `encoding='utf-8-sig'` 이유 설명 완료
- [ ] 심화 도전 과제 안내 완료

**다음 수업 연계**: 이 CSV를 입력 데이터로 사용하여
머신러닝 흥행 예측 모델(회귀/분류)을 구축하는 실습으로 이어질 수 있습니다.
